In [ ]:
import numpy as np
import sisl
import matplotlib.pyplot as plt
from scipy import linalg as lin
from scipy import sparse as spa
from scipy.sparse import linalg as spla
from ase.visualize import view
from tqdm.auto import tqdm
from sisl import Hamiltonian
from numba import njit

In [ ]:
gr = sisl.geom.graphene()
Ham0 = Hamiltonian(gr)

bond = 1.43
r = (0.1*bond, 1e-2+bond)
t = (0.0, -2.7)
Ham0.construct([r,t])
Ham = Ham0.tile(1,0).tile(1,1)

In [ ]:
def reorder(parent):
    if not isinstance(parent, sisl.RealSpaceSE):
        raise TypeError("`parent`must be of type `sisl.RealsapceSE`.")
    H = parent.real_space_parent()
    _, elec_indices = parent.real_space_coupling(True)
    nC = len(elec_indices)
    
    all_atoms = np.arange(0, H.na)
    inside_atoms = np.delete(arr=all_atoms, # parent array to remove indices from
                             obj=elec_indices, # indices to remove 
                             axis=0 # if None will flatten `arr` before removal
                            )
    atoms_list = np.concatenate([elec_indices, inside_atoms])
    H_reorder = H.sub(atoms_list)
    H_reorder.reduce()
    H_reorder.set_nsc([1,1,1])
    return H_reorder, atoms_list, nC

def sHam(parent, tile, nk1, eta):
    """Method for calculating the surrounding (real-space) self-energy Hamiltonian for a tiled parent structure.

    Parameters
    ----------
    Parent : sisl.Geometry
        a physical object from which to calculate the real-space self-energy.
        The parent object *must* have only 3 supercells along the direction where
        self-energies are used.
    tile : tuple, int
        How many times to tile in semi-inf and transverse direction. If an integer is used, will use this value for both.
    nk1 : int
        Number of samples in k-space along transverse direction 
    eta : float or complex
        perturbation used for self-energy calculations

    Returns
    -------
    H_reorder : sisl.Hamiltonian
        Hamiltonian reordered to have the electrode as the first `nC` elements
    rse : sisl.RealSpaceSE
        The Real-space self-energy object used for calculations
    nC : int
        Number of atoms in electrode.
    """
    if not isinstance(eta, complex):
        eta = eta*1j # ensure the format is a complex number
    if isinstance(tile, tuple): # extract the tiling along semi-infinite and transverse direction
        Na = tile[0]
        Nb = tile[1]
    elif isinstance(tile, int):
        Na = Nb = tile
    
    # define the Realspace SE based on primitive unit-cell Hamiltonian tiled by Na x Nb
    rse = sisl.RealSpaceSE(parent=parent, # Hamiltonian the method inherets
                           semi_axis=0, # semi-infinite direction (in unit-cell directions A, B, C)
                           k_axes=1, # k-sampling direction (transverse to semi-inf)
                           unfold=(Na, Nb, 1) # how to tile/unfold the inhereted structure in A, B, C directions
                           ) 
    rse.setup(eta=np.imag(eta), bz=sisl.MonkhorstPack(parent=parent, nkpt=[1, nk1, 1])) # setup the Realspace Self-energy to use eta and `nk1`-points in around gamma point in the transverse direction.
    
    H_reorder, alist, nC = reorder(rse)
    return H_reorder, rse, alist, nC

def _infer_solver(M):
    if spa.isspmatrix(M):
        lu = spla.splu(M)
        solver = lambda e: lu.solve(e)
    else:
        lu, piv = lin.lu_factor(M)
        solver = lambda e: lin.lu_solve((lu, piv), e)
    return solver

def _diag_of_inv(M):
    return np.diag(lin.inv(M))
    K = len(M)
    diag = np.zeros(K, dtype=complex)
    
    solver = _infer_solver(M)
    for i, idx in enumerate(tqdm(np.arange(K), desc="Sparse diagonal solves", leave=False)):
        ei = np.zeros(K)
        ei[idx] = 1.0
        xi = solver(ei)
        diag[i] = xi[idx]
    return diag

def _calc_invG(z, RSE, H_sub, S_sub, alist):
    invG = S_sub*z - H_sub
    invG[0:len(alist), 0:len(alist)] -= RSE
    return invG

def _ldos(G):
    if (G.ndim == 1): diagG = G
    elif (G.ndim == 2): diagG = np.diag(G)
    else: raise ValueError(f"The input to {_ldos.__name__} must be either GF or its diagonal;\n{ G.ndim = }\n {G.shape = }")
    return -(1.0/np.pi)*np.imag(diagG)

def _reorder_rse(RSE, alist):
    return RSE[np.ix_(alist, alist)]

def _loop_ldos(z, H_reordered, RSE_reordered, alist):
    H = H_reordered.Hk(format="array")
    S = H_reordered.Sk(format="array")
    invG = _calc_invG(z, RSE_reordered, H, S, alist)
    diagG = _diag_of_inv(invG)
    return _ldos(diagG)

def _calc_ldos(energies, rse):
    HamNN_reorder, alist, nC = reorder(rse)
    eta = rse._options["eta"]
    N = len(HamNN_reorder)
    if isinstance(eta, complex):
        eta = np.imag(eta)
    elif isinstance(eta, float):
        pass
    else:
        raise ValueError(f"the eta used for RSE is invalid ({type(eta) = }; {eta = })")
    
    LDOS = np.zeros((len(energies), N), dtype=float)
    for iE, E in enumerate(tqdm(energies, desc="LDOS", leave=True)):
        z = E + eta*1j
        RSE = rse.self_energy(z)
        RSE_reorder = _reorder_rse(RSE, alist)
        LDOS[iE, :] = _loop_ldos(z, HamNN_reorder, RSE_reorder, alist)
    
    return LDOS

@njit
def _calc_dos(ldos):
    return np.sum(ldos, axis=1)

In [ ]:
Na = 12
Nb = 12
eta = 1e-2j
nk1 = int(np.ceil(3*900/Nb))

rse = sisl.RealSpaceSE(Ham, 0, 1, (Na, Nb, 1))
rse.setup(eta= 0.001, bz=sisl.MonkhorstPack(Ham, [1, nk1, 1]))

H, rse, alist, nC = sHam(parent=Ham, nk1=nk1, tile=Na, eta=eta)


In [ ]:
dE = 0.1
Emax = 3.0
Emin = -Emax
energies = np.arange(Emin, Emax+dE, dE)

LDOS = _calc_ldos(energies, rse)
LDOS = LDOS[:, nC:]
DOS = LDOS.sum(axis=1)

In [ ]:
DATA = np.load("alans_calcs.npz")
DOS_OG = DATA["dos"]

# === plot DOS ===
E_idx=31
E_idx=min(E_idx, len(energies)) -1

fig, ax = plt.subplots(1, 2, figsize=(6,4))

ax[0].plot(energies, DOS_OG, c="k", label="OG method")
ax[0].plot(energies, DOS, c="r", label="new method")
ax[0].vlines(x=energies[E_idx],ymin=0,ymax=600,color='k',linestyle='dashed')


for a in ax:
    a.set_xlabel("Energy (eV)")
    a.set_ylabel("DOS (states / eV)")
    a.set(ylim=(0,100), xlim=(-3,3))
    a.legend()

ax[1].plot(energies, np.abs(DOS_OG - DOS), label="Diff")
fig.tight_layout()
None